## Тематическое моделирование (Topic Modeling) в Google Colab

Тематическое моделирование позволяет выявить скрытые "темы" в коллекции документов. В этом задании вы примените тематическое моделирование методом LDA к отзывам из самого первого нашего датасета `Lesson_1_user_requests.csv` для определения основных тем, которые обсуждаются гражданами. (Датасет ищите в репозитории в папке Text_analysis_RUT/seminars
/seminar_1/)

1.  **Предобработка текста:** Стандартная предварительная обработка текстов с лемматизацией. Лемматизация может улучшить качество тематического моделирования.
2.  **Извлечение признаков:** Создаём словарь и корпус документов в формате, подходящем для выбранного алгоритма тематического моделирования (bag-of-words для LDA).
3.  **Применение алгоритма:** Применим алгоритм тематического моделирования  Латентное размещение Дирихле (LDA), используя библиотеки `sklearn` и `gensim`.
4.  **Определение количества тем:** Попробуйте определить оптимальное количество тем. Для начала выберите разумное фиксированное количество тем (от 5 до 15).
5.  **Интерпретация тем:** Проанализируйте полученные темы, изучив наиболее часто встречающиеся слова в каждой теме. Присвойте каждой теме краткое осмысленное название и описание на основе ее ключевых слов.



In [ ]:
!pip install pymorphy3

In [ ]:
!pip install gensim

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
import pymorphy3
from tqdm.auto import tqdm
import re
import matplotlib.pyplot as plt
from sklearn.decomposition import LatentDirichletAllocation as LDA
from sklearn.feature_extraction.text import CountVectorizer
import seaborn as sns
from gensim.corpora import Dictionary
from gensim.models import LdaModel

nltk.download('stopwords')

In [ ]:
df_text = pd.read_csv('/content/drive/MyDrive/Lesson_1_user_requests.csv')

In [ ]:
df_text

Удалим пропущенные значения и дубликаты. Оставим только сами тексты обращений, а также сферу и категорию для каждого обращения, чтобы у нас был ориентир по ожидаемому количеству возможных тем.

In [ ]:
df_text = df_text.dropna(subset = ['process_texts'])

duplicated_mask = df_text.duplicated(subset = 'process_texts', keep = False)
df_text = df_text.drop_duplicates(subset = 'process_texts', keep = 'first')

df_text = df_text[['process_texts', 'sphera', 'categoriya']]

print("Всего строк в таблице: ", df_text.shape[0])

In [ ]:
df_text

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(y='sphera', data=df_text)
plt.title('Распределение по сферам')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(y='categoriya', data=df_text)
plt.title('Распределение по категориям')
plt.show()

In [ ]:
# Уберем строки, относящиеся к редко встречающимся классам из 'sphera' и 'categoriya'
# Например классы, которые встречаются реже 500 раз
sphera_counts = df_text['sphera'].value_counts()
spheras_to_keep = sphera_counts[sphera_counts >= 500].index
df_filtered = df_text[df_text['sphera'].isin(spheras_to_keep)]

categoriya_counts = df_filtered['categoriya'].value_counts()
categoriya_to_keep = categoriya_counts[categoriya_counts >= 500].index
df_filtered = df_filtered[df_filtered['categoriya'].isin(categoriya_to_keep)]

print(f"Оставлено строк после фильтрации : {df_filtered.shape[0]}")
display(df_filtered.head())

In [ ]:
df_filtered['word_count'] = df_filtered['process_texts'].apply(lambda x: len(x.split()))

plt.figure(figsize=(10, 6))
plt.hist(df_filtered['word_count'], bins=100, color='blue', edgecolor='black')
plt.title('Количество слов в каждом отзыве')
plt.xlabel('Количество слов')
plt.ylabel('Частота');

In [ ]:
# Уберем строки, где 'word_count' больше 400
df_filtered = df_filtered[df_filtered['word_count'] <= 400]

print(f"Оставлено строк после фильтрации : {df_filtered.shape[0]}")
display(df_filtered.head())

Проводим стандартную предобработку текста с лемматизацией: удаляем лишние символы (всё кроме букв и пробелов), приводим слова к нижнему регистру, удаляем стоп-слова, и проводим лемматизацию.

Также мы создаём структуру из вложенных списков (списки отдельных токенов из каждого текста являются элементами одного большого списка) для использования реализации LDA в библиотеке `gensim`.

In [ ]:
# создаём морфологический анализатор
morph = pymorphy3.MorphAnalyzer()

if 'russian_stopwords' not in locals():
      russian_stopwords = stopwords.words("russian")

texts_as_lists = []

def preprocess_text(text):
    if isinstance(text, str):
        text = re.sub(r'[^а-яё]', ' ', text) # Удаление пунктуации, кроме пробелов
        # разбиваем на слова
        text = text.lower() # Приведение к нижнему регистру
        words = text.split()
        # приводим к леммам, убираем стоп-слова и короткие слова (это важно для тематическго моделирования)
        lemmas = [morph.parse(word)[0].normal_form for word in words if word not in russian_stopwords and len(word) > 2]
        texts_as_lists.append(lemmas)
        return " ".join(lemmas)
    return "" # Возвращаем пустую строку для нестроковых значений


print("\nНачало предобработки текста...")
# Применение предобработки к столбцу с текстом. Используем tqdm для отслеживания прогресса.
if 'process_texts' in df_filtered.columns:
    tqdm.pandas()
    df_filtered['processed_text'] = df_filtered['process_texts'].progress_apply(preprocess_text)
    print("Предобработка текста завершена.")
    display(df_filtered.head())

Для векторизации используем CountVectorizer, а не TfidfVectorizer, так как для LDA необходимо истинное количество появлений каждого слова в документах, а TfidfVectorizer намеренно искажает частотность, придавая больший вес словам, которые часто встречаются в одном тексте и редко в остальных, что потенциально снижает точность LDA-модели.

Также для расчетов LDA мы не будем использовать эмбеддинги типа Word2Vec и BERT. LDA — это вероятностная модель, которая ожидает на вход матрицу слова-документы, состоящую из простого подсчета количества слов в документе. А в эмбеддингах заложена дополнительная информация о контексте и порядке слов.

Далее нам необходимо уменьшить размерность получившейся матрицы. Это можно сделать через PCA (Principal Component Analysis)/SVD (Singular Value Decomposition), а можно сразу ограничить словарь через параметры CountVectorizer:

* параметр max_df позволяет исключить слова, которые встречаются почти в каждом документе и поэтому, скорее всего, не несут значимую тематическую информацию;

* параметр min_df помогает отсеять слова, которые встречаются очень редко, а потому не дают важной информации, но расширяют словарь.

Предварительная обработка текста позволит нам уменьшить необходимый объем оперативной памяти и существенно сократить время, необходимое на обучение модели LDA.

In [ ]:
# Векторизация текста
count_vectorizer = CountVectorizer(max_features=1000, ngram_range=(1, 1), stop_words=russian_stopwords, max_df=0.9, min_df=2)
dataset = count_vectorizer.fit_transform(df_filtered['processed_text'])

In [ ]:
# Задаём модель LDA для 5 тем
lda = LDA(n_components = 5,
         max_iter=10,
         n_jobs=-1,
         learning_method='batch',
         random_state=42)
lda.fit(dataset)

In [ ]:
# Функция для визуализации 10 самых значимых слов в каждой теме
def display_topics(model, feature_names, no_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print(f"Тема {topic_idx + 1}:", ", ".join([feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]))

In [ ]:
tf_feature_names = count_vectorizer.get_feature_names_out()
display_topics(lda, tf_feature_names, 10)


Мы можем условно обозначить Тему 3 как "детские площадки и двор", а Тему 5 как "отопление".

Попробуем увеличить количество тем в 2 раза.

In [ ]:
lda = LDA(n_components = 10,
         max_iter=10,
         n_jobs=-1,
         learning_method='batch',
         random_state=42)
lda.fit(dataset)

In [ ]:
tf_feature_names = count_vectorizer.get_feature_names_out()
display_topics(lda, tf_feature_names, 10)

Обозначим наиболее очевидные из полученных тем:

Тема 2: "Общественный транспорт"

Тема 3: "Детские площадки"

Тема 5: "Оплата ЖКУ"

Тема 9: "Уличное освещение"

При использовании LDA и других методов тематического моделирования важно экспериментировать с различными параметрами (в частности количеством тем) и методами предобработки данных. Также имейте в виду, что интерпретация результатов может потребовать дополнительного анализа и контекста.

Рассмотрим также пример тематического моделирования с использованием LDA из библиотеки `gensim`.

In [ ]:
# Предобработка текста для тематического моделирования в gensim
# Для тематического моделирования часто полезно удалить очень редкие и очень частые слова

# Создание словаря
gensim_dictionary = Dictionary(texts_as_lists)

# Фильтрация слов: удаление очень редких (меньше 5 документов) и очень частых (встречаются более чем в 50% документов)
gensim_dictionary.filter_extremes(no_below=5, no_above=0.5)

# Создание корпуса (мешок слов)
corpus = [gensim_dictionary.doc2bow(text, allow_update=True) for text in texts_as_lists]

print(f"Создан словарь с {len(gensim_dictionary)} уникальными токенами.")
print(f"Создан корпус из {len(corpus)} документов.")

In [ ]:
# Применение алгоритма LDA
# Выбираем количество тем.
num_topics = 10 # Например, 10 тем

print(f"\nОбучение модели LDA с {num_topics} темами...")
# Обучение LDA модели
lda_model = LdaModel(corpus, num_topics=num_topics, id2word=gensim_dictionary, passes=15) # Увеличиваем passes для лучшей сходимости

print("Обучение LDA модели завершено.")

# Интерпретация тем
print("\nВыявленные темы:")
topics = lda_model.print_topics(num_words=10) # Выводим 10 наиболее значимых слов для каждой темы

for topic_id, topic_words in topics:
    print(f"Тема #{topic_id + 1}: {topic_words}")

Для решения задачи тематического моделирования можно также использовать трансформеры и большие языковые модели (LLM). Интересный пример использования в анализе отзывов о российских банках можно найти здесь https://habr.com/ru/companies/gazprombank/articles/909406/

Разные методы тематического моделирования имеют свои особенности в решении различных задач:

* **LDA**: Хорошо подходит для выявления скрытых тем и их распределения в коллекции. Если вам важно понять, какие темы присутствуют в документах и как они связаны друг с другом, LDA может быть предпочтительным выбором.

* **NMF**: Неотрицательная матричная факторизация (NMF) — это метод анализа данных, который находит разложение неотрицательной матрицы на две также неотрицательные матрицы меньшей размерности. Этот метод особенно полезен, когда неотрицательность данных имеет смысл. Например, в анализе отзывов, где оценки являются неотрицательными числами, NMF может дать более интерпретируемые результаты. (`sklearn.decomposition.NMF`)

* **pLSA**: pLSA подходит, если вы хотите понять, какие темы и слова связаны с конкретными документами. Однако, имейте в виду, что pLSA требует больше вычислительных ресурсов и может быть менее стабильным по сравнению с LDA.

* **LSI**: Если ваша главная задача - снижение размерности данных и выделение семантических структур, LSI может быть полезным. (`gensim.models.LsiModel`)

* **Кластерный анализ**: Если вы хотите просто разделить документы на группы схожих по тематике текстов, кластерный анализ может быть эффективным подходом. (`sklearn.cluster.KMeans`)

# Задания:
1) Используйте coherence score (gensim.models.coherencemodel.CoherenceModel) для поиска оптимального числа тем
2) По наиболее значимым словам попробуйте предложить свою интерпретацию для каждой из полученных тем
3) Протестируйте модели LsiModel, NMF
4) Попробуйте модифицировать список стоп-слов для улучшения результатов тематического моделирования